In [1]:
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent
events_filepath = BASE_DIR / "data/raw/events_data"
output_path = BASE_DIR / "data/clean/events.parquet"

In [3]:
events_data = pd.read_parquet(output_path)

In [4]:
tour_filepath = BASE_DIR / "data/raw/tours_complete_info.csv"

dtypes = {
    "tour_id": "Int64",
    "tour_name":"string",
    "tour_length":"Int64",
    "total_stories":"Int64"
}
tour_info = pd.read_csv(tour_filepath, dtype=dtypes)


In [6]:
tour_info

,tour_id,tour_name,tour_length,total_stories
0,51,Ancient Agora: the birth of democracy,140,113
1,55,National Archaeological Museum: the notable Gr...,100,82
2,79,The curious Oltrarno,<NA>,19
3,86,"Delphi and Iera Chora, the steps of Parnassus",180,13
4,91,Piraeus: Hidden urban stories,110,51
...,...,...,...,...
172,898,﻿The Lyceum of Aristotle: Talking about philos...,20,12
173,903,Plaka City Tour: The neighborhood of the gods,<NA>,<NA>
174,908,Julia Demo Jorge: Washington D.C. City Tour,<NA>,<NA>
175,909,Julia Demo Maria: Plaka City Tour,<NA>,<NA>


In [7]:
story_filepath = BASE_DIR / "data/raw/story_id_details.csv"

dtypes = {
    "tour_id": "Int64",
    "tour_item_id":"Int64",
    "story_id":"Int64",
    "total_item_title":"string",
    "story_title":"string"
}
stories = pd.read_csv(story_filepath, dtype=dtypes)


In [8]:
stories

,tour_id,tour_item_id,story_id,tour_item_title,story_title
0,1,9,5,Lambros Eftaxias,Lambros Eftaxias (1915-1996)
1,1,9,347,Lambros Eftaxias,His career
2,1,9,348,Lambros Eftaxias,The collector
3,1,9,349,Lambros Eftaxias,Donations to Museums
4,1,9,350,Lambros Eftaxias,The Friends of Music Association (1953)
...,...,...,...,...,...
33645,912,15060,56240,Stop 20: Columbus,Stop 20: Columbus
33646,912,15061,56241,Old Customs House,Old Customs House
33647,912,15061,56242,Old Customs House,Pre Olympics
33648,912,15061,56243,Old Customs House,Port Vell Cable Car


In [9]:
# 1️⃣ Stories appearing in events
events_story_counts = (
    events_data
    .dropna(subset=["tour_id", "story_id"])
    .groupby("tour_id")["story_id"]
    .nunique()
    .reset_index()
    .rename(columns={"story_id": "stories_in_events"})
)

# 2️⃣ Stories defined in story_details csv
stories_csv_counts = (
    stories
    .groupby("tour_id")["story_id"]
    .nunique()
    .reset_index()
    .rename(columns={"story_id": "stories_in_story_details"})
)

# 3️⃣ Stories shown on website (tour_info csv)
website_counts = (
    tour_info[["tour_id", "total_stories"]]
    .rename(columns={"total_stories": "stories_in_website"})
)

# 4️⃣ Merge everything
comparison_table = (
    stories_csv_counts
    .merge(events_story_counts, on="tour_id", how="left")
    .merge(website_counts, on="tour_id", how="left")
)

# Optional: sort for easier inspection
comparison_table = comparison_table.sort_values("tour_id")


In [11]:
comparison_table = comparison_table.dropna(subset=["stories_in_events"])

comparison_table = comparison_table.reset_index(drop=True)

In [12]:
comparison_table

,tour_id,stories_in_story_details,stories_in_events,stories_in_website
0,51,243,113.0,113
1,55,154,82.0,82
2,79,20,19.0,19
3,83,19,9.0,<NA>
4,86,17,13.0,13
...,...,...,...,...
183,898,42,12.0,12
184,903,56,53.0,<NA>
185,908,67,67.0,<NA>
186,909,53,53.0,<NA>


In [18]:
# rows where website count exists AND differs from story_details
diff = comparison_table[
    comparison_table["stories_in_website"].notna()
    & (comparison_table["stories_in_story_details"] != comparison_table["stories_in_website"])
]

diff.shape[0], comparison_table["stories_in_website"].notna().sum()

(59, 145)

In [20]:
diff

,tour_id,stories_in_story_details,stories_in_events,stories_in_website
0,51,243,113.0,113
1,55,154,82.0,82
2,79,20,19.0,19
4,86,17,13.0,13
6,107,220,77.0,83
9,139,60,23.0,50
11,181,57,31.0,31
12,226,118,78.0,78
13,239,31,25.0,30
14,240,83,80.0,80


## Count of story_id in events data

In [19]:
story_event_counts = (
    events_data
    .dropna(subset=["tour_id", "story_id"])
    .groupby(["tour_id", "story_id"])
    .size()
    .reset_index(name="event_count")
    .sort_values(["tour_id", "story_id"])
)

story_event_counts

,tour_id,story_id,event_count
0,51,9635,3471
1,51,9636,3664
2,51,9637,3045
3,51,9638,3216
4,51,9639,2910
...,...,...,...
9627,909,55596,3
9628,910,55597,22
9629,910,55598,17
9630,910,55599,4


In [21]:
stories

,tour_id,tour_item_id,story_id,tour_item_title,story_title
0,1,9,5,Lambros Eftaxias,Lambros Eftaxias (1915-1996)
1,1,9,347,Lambros Eftaxias,His career
2,1,9,348,Lambros Eftaxias,The collector
3,1,9,349,Lambros Eftaxias,Donations to Museums
4,1,9,350,Lambros Eftaxias,The Friends of Music Association (1953)
...,...,...,...,...,...
33645,912,15060,56240,Stop 20: Columbus,Stop 20: Columbus
33646,912,15061,56241,Old Customs House,Old Customs House
33647,912,15061,56242,Old Customs House,Pre Olympics
33648,912,15061,56243,Old Customs House,Port Vell Cable Car


In [ ]:
# 1️⃣ Keep only tours that appear in events
stories_filtered = stories[
    stories["tour_id"].isin(events_story_counts["tour_id"])
].copy()

# 2️⃣ Create (tour_id, story_id) key for fast lookup
event_story_pairs = set(
    zip(story_event_counts["tour_id"], story_event_counts["story_id"])
)

# 3️⃣ Create in_events column
stories_filtered["in_events"] = stories_filtered.apply(
    lambda row: (row["tour_id"], row["story_id"]) in event_story_pairs,
    axis=1
)


,tour_id,tour_item_id,story_id,tour_item_title,story_title,in_events
2271,55,676,2471,Sea of Fire | Thera,The Boxing children fresco (ΒΕ 974.26),False
2272,55,676,2472,Sea of Fire | Thera,The Spring fresco,False
2273,55,676,2473,Sea of Fire | Thera,Plaster cast of a bed,False
2274,55,676,2474,Sea of Fire | Thera,Carbonized seeds (ΒΕ 1974.25),False
2275,55,676,6555,Sea of Fire | Thera,Map to the exhibitions,False
...,...,...,...,...,...,...
33050,910,14859,55643,Dom Luis I bridge,"The siege, the panic, the disaster",False
33051,910,14859,55644,Dom Luis I bridge,With a touch of Eiffel Tower,False
33052,910,14859,55645,Dom Luis I bridge,A bridge to the wineries,False
33053,910,14859,55646,Dom Luis I bridge,Tip,False


In [24]:
stories_filtered = stories_filtered.sort_values(
    ["tour_id", "story_id"],
    ascending=[True, True]
).reset_index(drop=True)

stories_filtered

,tour_id,tour_item_id,story_id,tour_item_title,story_title,in_events
0,51,687,2530,The Boule of the 500,Cleisthenes and the Boule of the 500,False
1,51,687,2531,The Boule of the 500,The Bouleuterion,False
2,51,687,2532,The Boule of the 500,Joining the Boule,False
3,51,687,2533,The Boule of the 500,Taking the Oath,False
4,51,687,2534,The Boule of the 500,Broadcasting the Boule,False
...,...,...,...,...,...,...
14085,910,14859,55643,Dom Luis I bridge,"The siege, the panic, the disaster",False
14086,910,14859,55644,Dom Luis I bridge,With a touch of Eiffel Tower,False
14087,910,14859,55645,Dom Luis I bridge,A bridge to the wineries,False
14088,910,14859,55646,Dom Luis I bridge,Tip,False


In [25]:
close_match_tours = diff[
    (diff["stories_in_website"].notna()) &
    ( (diff["stories_in_events"] - diff["stories_in_website"]).abs() <= 1 )
]["tour_id"]

close_match_tours.tolist()

[51,
 55,
 79,
 86,
 181,
 226,
 240,
 278,
 285,
 309,
 334,
 375,
 415,
 416,
 434,
 447,
 456,
 460,
 490,
 507,
 515,
 520,
 523,
 550,
 552,
 619,
 641,
 821,
 898]

In [26]:
diff[
    (diff["stories_in_website"].notna()) &
    ( (diff["stories_in_events"] - diff["stories_in_website"]).abs() <= 1 )
]

,tour_id,stories_in_story_details,stories_in_events,stories_in_website
0,51,243,113.0,113
1,55,154,82.0,82
2,79,20,19.0,19
4,86,17,13.0,13
11,181,57,31.0,31
12,226,118,78.0,78
14,240,83,80.0,80
19,278,112,105.0,105
23,285,61,60.0,60
30,309,57,53.0,52


In [27]:
# Convert to set for faster lookup
close_match_set = set(close_match_tours)

# Keep:
# - All tours NOT in close_match_set (unchanged)
# - For tours in close_match_set → keep only in_events == True
stories_filtered = stories_filtered[
    (~stories_filtered["tour_id"].isin(close_match_set)) |
    (stories_filtered["in_events"] == True)
].reset_index(drop=True)

stories_filtered

,tour_id,tour_item_id,story_id,tour_item_title,story_title,in_events
0,51,2933,9635,Entrance,Directions,True
1,51,2933,9636,Entrance,A place to gather?,True
2,51,2933,9637,Entrance,The golden age,True
3,51,2933,9638,Entrance,The rise and fall,True
4,51,2933,9639,Entrance,Back to life,True
...,...,...,...,...,...,...
13093,910,14859,55643,Dom Luis I bridge,"The siege, the panic, the disaster",False
13094,910,14859,55644,Dom Luis I bridge,With a touch of Eiffel Tower,False
13095,910,14859,55645,Dom Luis I bridge,A bridge to the wineries,False
13096,910,14859,55646,Dom Luis I bridge,Tip,False


In [28]:
stories_filtered = stories_filtered.sort_values(
    ["tour_id", "tour_item_id", "story_id"],
    ascending=[True, True, True]
).reset_index(drop=True)

In [30]:
stories_filtered["story_position"] = (
    stories_filtered.groupby("tour_id").cumcount() + 1
)

In [31]:
stories_filtered

,tour_id,tour_item_id,story_id,tour_item_title,story_title,in_events,story_position
0,51,2933,9635,Entrance,Directions,True,1
1,51,2933,9636,Entrance,A place to gather?,True,2
2,51,2933,9637,Entrance,The golden age,True,3
3,51,2933,9638,Entrance,The rise and fall,True,4
4,51,2933,9639,Entrance,Back to life,True,5
...,...,...,...,...,...,...,...
13093,910,14859,55643,Dom Luis I bridge,"The siege, the panic, the disaster",False,47
13094,910,14859,55644,Dom Luis I bridge,With a touch of Eiffel Tower,False,48
13095,910,14859,55645,Dom Luis I bridge,A bridge to the wineries,False,49
13096,910,14859,55646,Dom Luis I bridge,Tip,False,50


## What if only android?

In [33]:
# 1️⃣ Filter events to Android only
events_android = events_data[
    events_data["platform"] == "ANDROID"
]

# 2️⃣ Stories appearing in Android events
events_story_counts_android = (
    events_android
    .dropna(subset=["tour_id", "story_id"])
    .groupby("tour_id")["story_id"]
    .nunique()
    .reset_index()
    .rename(columns={"story_id": "stories_in_events_android"})
)

# 3️⃣ Stories defined in story_details csv
stories_csv_counts = (
    stories
    .groupby("tour_id")["story_id"]
    .nunique()
    .reset_index()
    .rename(columns={"story_id": "stories_in_story_details"})
)

# 4️⃣ Website info
website_counts = (
    tour_info[["tour_id", "total_stories"]]
    .rename(columns={"total_stories": "stories_in_website"})
)

# 5️⃣ Merge
comparison_android = (
    stories_csv_counts
    .merge(events_story_counts_android, on="tour_id", how="left")
    .merge(website_counts, on="tour_id", how="left")
)

comparison_android

,tour_id,stories_in_story_details,stories_in_events_android,stories_in_website
0,1,117,NaN,<NA>
1,2,124,NaN,<NA>
2,3,45,NaN,<NA>
3,5,18,NaN,<NA>
4,8,37,NaN,<NA>
...,...,...,...,...
776,917,28,NaN,<NA>
777,918,47,NaN,<NA>
778,919,75,NaN,<NA>
779,920,13,NaN,<NA>


In [35]:
comparison_android = comparison_android.dropna(subset=["stories_in_events_android"])

comparison_android = comparison_android.reset_index(drop=True)

In [36]:
comparison_android

,tour_id,stories_in_story_details,stories_in_events_android,stories_in_website
0,51,243,113.0,113
1,55,154,82.0,82
2,79,20,11.0,19
3,91,51,38.0,51
4,107,220,77.0,83
...,...,...,...,...
131,897,50,13.0,<NA>
132,898,42,8.0,12
133,908,67,67.0,<NA>
134,909,53,53.0,<NA>


In [37]:
# Keep only relevant columns
events_all = comparison_table[["tour_id", "stories_in_events"]]

events_android = comparison_android[["tour_id", "stories_in_events_android"]]

# Merge side by side
side_by_side = (
    events_all
    .merge(events_android, on="tour_id", how="inner")
    .sort_values("tour_id")
    .reset_index(drop=True)
)

side_by_side

,tour_id,stories_in_events,stories_in_events_android
0,51,113.0,113.0
1,55,82.0,82.0
2,79,19.0,11.0
3,91,51.0,38.0
4,107,77.0,77.0
...,...,...,...
131,897,16.0,13.0
132,898,12.0,8.0
133,908,67.0,67.0
134,909,53.0,53.0


In [ ]:
side_by_side["difference"] = (
    side_by_side["stories_in_events"] -
    side_by_side["stories_in_events_android"]
)


In [39]:
side_by_side.describe()

,tour_id,stories_in_events,stories_in_events_android,difference
count,136.0,136.000000,136.000000,136.000000
mean,571.323529,57.617647,50.852941,6.764706
std,232.04367,31.914715,29.855337,17.107605
min,51.0,1.000000,1.000000,0.000000
25%,436.75,39.000000,30.500000,0.000000
50%,538.5,55.000000,52.000000,0.000000
75%,815.5,76.000000,72.250000,6.250000
max,910.0,148.000000,138.000000,130.000000
